<a href="https://colab.research.google.com/github/zeosapka/glass-box-ai-research/blob/main/notebooks/13_local_llm_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# E12 — Local LLM / Ollama: Ortam ve Deney Ayarları
# Amaç: Yerel LLM deneyinin temel ayarlarını tanımlamak ve
#       Ollama'nın Colab ortamında kullanılabilir olup olmadığını
#       kontrol etmek.
# ============================================================

import os
import platform
import subprocess
import sys

# Kullanacağımız küçük yerel dil modeli.
# 1B parametreli olması, E12'nin "küçük local LLM" şartına uygundur.
MODEL_NAME = "llama3.2:1b"

# Deneyde kullanacağımız üç farklı prompt.
# Amaç: modelin farklı türde sorulara verdiği çıktıları gözlemlemek.
PROMPTS = [
    {
        "id": "P1",
        "type": "factual",
        "text": "What is a transformer model in artificial intelligence? Explain briefly."
    },
    {
        "id": "P2",
        "type": "reasoning",
        "text": "Why might a language model give different answers to two very similar questions?"
    },
    {
        "id": "P3",
        "type": "glass_box",
        "text": "What kinds of internal information might a language model use when generating an answer?"
    }
]

print("=== E12 Local LLM Experiment ===")
print(f"Python version : {sys.version.split()[0]}")
print(f"Operating system: {platform.system()}")
print(f"Target model   : {MODEL_NAME}")
print(f"Prompt count   : {len(PROMPTS)}")
print()

# Ollama'nın sistemde kurulu olup olmadığını kontrol ediyoruz.
# Kurulu değilse hata vermek yerine bunu açıkça raporluyoruz.
try:
    result = subprocess.run(
        ["ollama", "--version"],
        capture_output=True,
        text=True
    )

    print("Ollama durumu:")
    print(result.stdout.strip() or result.stderr.strip())

except FileNotFoundError:
    print("Ollama henüz bu ortamda kurulu değil.")

=== E12 Local LLM Experiment ===
Python version : 3.13.15
Operating system: Linux
Target model   : llama3.2:1b
Prompt count   : 3

Ollama henüz bu ortamda kurulu değil.


In [3]:
# ============================================================
# E12 — Ollama bağımlılığı
# Amaç: Ollama kurulumunun ihtiyaç duyduğu zstd paketini
#       Colab ortamına yüklemek.
# ============================================================

import subprocess

result = subprocess.run(
    ["apt-get", "update"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("apt update başarısız.")

result = subprocess.run(
    ["apt-get", "install", "-y", "zstd"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("zstd kurulumu başarısız.")

print("=" * 60)
print("zstd kurulumu tamamlandı.")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 15 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (9,396 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading

In [4]:
# ============================================================
# E12 — Ollama Kurulumu
# ============================================================

import subprocess

result = subprocess.run(
    [
        "bash",
        "-c",
        "curl -fsSL https://ollama.com/install.sh | sh"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Ollama kurulumu başarısız.")

version = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True
)

print("=" * 60)
print("Ollama kurulumu tamamlandı.")
print("Version:", version.stdout.strip())


Ollama kurulumu tamamlandı.
Version: Warning: could not connect to a running Ollama instance


In [5]:
# ============================================================
# E12 — Ollama Server Başlatma
# Amaç: Colab ortamında Ollama API/server sürecini başlatmak.
# ============================================================

import subprocess
import time
import requests

# Ollama server'ı arka planda başlat
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Server'ın ayağa kalkmasını bekle
time.sleep(5)

# Ollama API kontrolü
try:
    response = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=5
    )

    if response.status_code == 200:
        print("Ollama server: ÇALIŞIYOR")
        print("API: http://127.0.0.1:11434")
    else:
        print("Ollama server yanıt verdi ancak durum kodu:")
        print(response.status_code)

except Exception as e:
    print("Ollama server'a bağlanılamadı.")
    print("Hata:", e)

Ollama server: ÇALIŞIYOR
API: http://127.0.0.1:11434


In [6]:
# ============================================================
# E12 — Local LLM Model Download
# Amaç: Llama 3.2 1B modelini Ollama üzerinden indirmek.
# ============================================================

import subprocess

MODEL_NAME = "llama3.2:1b"

print(f"Model indiriliyor: {MODEL_NAME}")
print("=" * 60)

result = subprocess.run(
    ["ollama", "pull", MODEL_NAME],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Model indirilemedi.")

print("=" * 60)
print(f"Model hazır: {MODEL_NAME}")

Model indiriliyor: llama3.2:1b

Model hazır: llama3.2:1b


In [7]:
# ============================================================
# E12 — Local LLM / 3 Prompt Test
# Amaç:
# Yerel Llama 3.2 1B modelinin farklı prompt türlerine verdiği
# cevapları kaydetmek.
# ============================================================

import requests
import json
import time

API_URL = "http://127.0.0.1:11434/api/generate"

results = []

print("E12 — Local LLM Prompt Testi")
print("=" * 60)

for prompt_info in PROMPTS:

    prompt_id = prompt_info["id"]
    prompt_type = prompt_info["type"]
    prompt_text = prompt_info["text"]

    print(f"\n[{prompt_id}] {prompt_type}")
    print("-" * 60)
    print("Prompt:", prompt_text)

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt_text,
        "stream": False,
        "options": {
            "temperature": 0.0
        }
    }

    start_time = time.time()

    response = requests.post(
        API_URL,
        json=payload,
        timeout=120
    )

    elapsed_time = time.time() - start_time

    response.raise_for_status()

    data = response.json()

    output_text = data.get("response", "").strip()

    results.append({
        "prompt_id": prompt_id,
        "prompt_type": prompt_type,
        "prompt": prompt_text,
        "response": output_text,
        "elapsed_seconds": elapsed_time
    })

    print("\nModel response:")
    print(output_text)

    print(
        f"\nResponse time: "
        f"{elapsed_time:.2f} s"
    )

print("\n" + "=" * 60)
print(f"E12 tamamlandı — {len(results)} prompt işlendi.")

E12 — Local LLM Prompt Testi

[P1] factual
------------------------------------------------------------
Prompt: What is a transformer model in artificial intelligence? Explain briefly.

Model response:
In artificial intelligence, a transformer model is a type of neural network architecture that's particularly well-suited for natural language processing tasks, such as text classification, language translation, and question-answering. It was introduced in 2017 by Vaswani et al. and has since become a popular choice in many NLP applications. The transformer model works by modeling sequences of words or characters as vectors in a high-dimensional space, allowing it to capture complex relationships between words and generate coherent outputs.

Response time: 26.55 s

[P2] reasoning
------------------------------------------------------------
Prompt: Why might a language model give different answers to two very similar questions?

Model response:
There are several reasons why a language mode

In [8]:
# ============================================================
# E12 — Glass Box Değerlendirmesi
# Amaç:
# Local LLM çıktılarının yalnızca gözlemsel olarak kaydedildiğini
# ve modelin kendi açıklamalarının gerçek iç mekanizma kanıtı
# olarak kabul edilmediğini açıkça belgelemek.
# ============================================================

e12_evaluation = {
    "model": MODEL_NAME,
    "prompt_count": len(results),
    "local_execution": True,
    "internal_intervention": False,
    "mechanistic_evidence": False,
    "glass_box_assessment": (
        "The local LLM was successfully executed and produced "
        "responses to three different prompt types. The outputs "
        "demonstrate observable behavioral variation, but the "
        "responses themselves do not constitute evidence of the "
        "model's actual internal mechanisms. In particular, "
        "self-reported explanations about internal information "
        "sources must not be treated as mechanistic evidence."
    )
}

print("E12 — Glass Box Değerlendirmesi")
print("=" * 60)

print(f"Model: {e12_evaluation['model']}")
print(f"Prompt count: {e12_evaluation['prompt_count']}")
print(
    f"Local execution: "
    f"{e12_evaluation['local_execution']}"
)
print(
    f"Internal intervention: "
    f"{e12_evaluation['internal_intervention']}"
)
print(
    f"Mechanistic evidence: "
    f"{e12_evaluation['mechanistic_evidence']}"
)

print("\nAssessment:")
print(e12_evaluation["glass_box_assessment"])

E12 — Glass Box Değerlendirmesi
Model: llama3.2:1b
Prompt count: 3
Local execution: True
Internal intervention: False
Mechanistic evidence: False

Assessment:
The local LLM was successfully executed and produced responses to three different prompt types. The outputs demonstrate observable behavioral variation, but the responses themselves do not constitute evidence of the model's actual internal mechanisms. In particular, self-reported explanations about internal information sources must not be treated as mechanistic evidence.
